In [1]:
!pip uninstall -y transformers
!pip install transformers==4.41.2
!pip uninstall -y peft

Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 56.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 64.1 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.7.1
    Uninstalling huggingface_hub-1.7.1:
      Successfully uninstalled huggingface_hub-1.7.1
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
Found existing installation: peft 0.18.1
Uninstalling peft-0.18.1:
  Successfully uninstalled peft-0.18.1


In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
# from sklearn.model_selection

!pip install transformers datasets scikit-learn -q

import numpy as np
import torch
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, classification_report

from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

In [4]:
df = pd.read_csv('/content/drive/MyDrive/balanced_dataset_12k.csv')

In [5]:
df.head()

,headline,label,confidence
0,"How JD(S) is quietly remaking its leadership team: As a veteran MLA is removed from core committee, ‘first family’ scions get key posts",sarcastic,0.65
1,"Waqf Bill: More than 12 hours of debate, and a washroom break that got Congress all wet",sarcastic,0.85
2,"India-U.S. interim trade deal: ‘Namaste Trump scored over Howdy Modi’, says Congress",sarcastic,0.75
3,"‘Richest’ CM Chandrababu Naidu’s draws almost entirely on one asset, Mamata records sharpest decline",sarcastic,0.45
4,Sunil Jakhar ‘resignation’: How ex-Congress leader struggled to find feet in BJP,sarcastic,0.45


In [6]:
df = df.drop(columns=['confidence'], axis = 1)

In [7]:
import re

def clean_text(text):
    text = str(text)

    # remove URLs
    text = re.sub(r"http\S+|www\S+", "", text)

    # remove HTML tags
    text = re.sub(r"<.*?>", "", text)

    # remove extra whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text


df['text'] = df['headline'].apply(lambda x: clean_text(x))

In [8]:
df.head()

,headline,label,text
0,"How JD(S) is quietly remaking its leadership team: As a veteran MLA is removed from core committee, ‘first family’ scions get key posts",sarcastic,"How JD(S) is quietly remaking its leadership team: As a veteran MLA is removed from core committee, ‘first family’ scions get key posts"
1,"Waqf Bill: More than 12 hours of debate, and a washroom break that got Congress all wet",sarcastic,"Waqf Bill: More than 12 hours of debate, and a washroom break that got Congress all wet"
2,"India-U.S. interim trade deal: ‘Namaste Trump scored over Howdy Modi’, says Congress",sarcastic,"India-U.S. interim trade deal: ‘Namaste Trump scored over Howdy Modi’, says Congress"
3,"‘Richest’ CM Chandrababu Naidu’s draws almost entirely on one asset, Mamata records sharpest decline",sarcastic,"‘Richest’ CM Chandrababu Naidu’s draws almost entirely on one asset, Mamata records sharpest decline"
4,Sunil Jakhar ‘resignation’: How ex-Congress leader struggled to find feet in BJP,sarcastic,Sunil Jakhar ‘resignation’: How ex-Congress leader struggled to find feet in BJP


In [9]:
from huggingface_hub import login
login(token="###########################")

In [10]:
texts = df["headline"].tolist()
labels = df["label"].tolist()

train_texts, val_texts, train_labels, val_labels = train_test_split(
    texts, labels, test_size=0.1, stratify=labels, random_state=42
)

train_dataset = Dataset.from_dict({"text": train_texts, "label": train_labels})
val_dataset   = Dataset.from_dict({"text": val_texts, "label": val_labels})

In [11]:
model_name = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(example):
    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=128   # 🔥 optimized for speed
    )

train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset   = val_dataset.map(tokenize, batched=True)

train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
val_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/10605 [00:00<?, ? examples/s]

Map:   0%|          | 0/1179 [00:00<?, ? examples/s]

In [12]:
config = AutoConfig.from_pretrained(
    model_name,
    num_labels=2,
    hidden_dropout_prob=0.3,
    attention_probs_dropout_prob=0.3
)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    config=config
)

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [13]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print("Using:", device)

Using: cuda


In [14]:
for param in model.roberta.encoder.layer[:6].parameters():
    param.requires_grad = False

In [15]:
training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/Colab Notebooks/major projectr/results_major",

    # 🔥 GPU optimization
    per_device_train_batch_size=128,
    per_device_eval_batch_size=128,
    fp16=False,

    # training control
    learning_rate=2e-5,
    num_train_epochs=50,
    warmup_ratio=0.1,
    weight_decay=0.01,

    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,

    load_best_model_at_end=True,
    metric_for_best_model="f1",

    report_to="none"
)

/usr/local/lib/python3.12/dist-packages/transformers/training_args.py:1474: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [16]:
def encode_labels(example):
    label_map = {
        "non-sarcastic": 0,
        "sarcastic": 1
    }
    example["label"] = label_map[example["label"]]
    return example

In [17]:
train_dataset = train_dataset.map(encode_labels)
val_dataset = val_dataset.map(encode_labels)

Map:   0%|          | 0/10605 [00:00<?, ? examples/s]

Map:   0%|          | 0/1179 [00:00<?, ? examples/s]

In [18]:
print(train_dataset[0])

{'label': tensor(0), 'input_ids': tensor([    0,   387, 12887,    40,   283,     7,   476,   114,  3346,  2834,
           32,  6408,    35,   392,  1584,  3648,     2,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,  

In [19]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    return {
        # Overall
        "accuracy": accuracy_score(labels, preds),

        # Binary class metrics (positive = sarcastic)
        "precision": precision_score(labels, preds),
        "recall": recall_score(labels, preds),
        "f1": f1_score(labels, preds),

        # Macro (treat classes equally)
        "precision_macro": precision_score(labels, preds, average="macro"),
        "recall_macro": recall_score(labels, preds, average="macro"),
        "f1_macro": f1_score(labels, preds, average="macro"),

        # Weighted (accounts for imbalance)
        "precision_weighted": precision_score(labels, preds, average="weighted"),
        "recall_weighted": recall_score(labels, preds, average="weighted"),
        "f1_weighted": f1_score(labels, preds, average="weighted"),
    }

# STEP 1 — FIX LABELS FIRST
# def encode_labels(example):
#     label_map = {
#         "non-sarcastic": 0,
#         "sarcastic": 1
#     }
#     example["label"] = label_map[example["label"]]
#     return example

# train_dataset = train_dataset.map(encode_labels)
# val_dataset = val_dataset.map(encode_labels)

# STEP 2 — REMOVE EXTRA COLUMNS (IMPORTANT)
train_dataset = train_dataset.remove_columns(
    [col for col in train_dataset.column_names if col not in ["input_ids", "attention_mask", "label"]]
)

val_dataset = val_dataset.remove_columns(
    [col for col in val_dataset.column_names if col not in ["input_ids", "attention_mask", "label"]]
)

# STEP 3 — SET FORMAT
train_dataset.set_format(type="torch")
val_dataset.set_format(type="torch")

# STEP 4 — NOW CREATE TRAINER (AFTER FIXES)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

# STEP 5 — TRAIN
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Precision Macro,Recall Macro,F1 Macro,Precision Weighted,Recall Weighted,F1 Weighted
1,0.699700,0.692172,0.514843,0.507398,0.989813,0.670886,0.653699,0.515246,0.374153,0.653823,0.514843,0.373901
2,0.695700,0.688114,0.654792,0.597430,0.947368,0.732764,0.735450,0.655040,0.622670,0.735567,0.654792,0.622576
3,0.689000,0.604216,0.675148,0.722944,0.567063,0.635585,0.683648,0.675057,0.671274,0.683614,0.675148,0.671304
4,0.608200,0.566134,0.697201,0.750000,0.590832,0.660969,0.706469,0.697111,0.693703,0.706432,0.697201,0.693730
5,0.599700,0.548320,0.715861,0.759184,0.631579,0.689527,0.722117,0.715789,0.713802,0.722086,0.715861,0.713823
6,0.583600,0.603343,0.681086,0.817910,0.465195,0.593074,0.722344,0.680903,0.665435,0.722263,0.681086,0.665496
7,0.566000,0.687454,0.648855,0.848606,0.361630,0.507143,0.721717,0.648612,0.617208,0.721609,0.648855,0.617301
8,0.560500,0.667593,0.653096,0.830882,0.383701,0.524971,0.715331,0.652868,0.625879,0.715233,0.653096,0.625965
9,0.554600,0.622061,0.683630,0.837500,0.455008,0.589659,0.731905,0.683436,0.666120,0.731815,0.683630,0.666185
10,0.551500,0.617218,0.687023,0.841615,0.460102,0.594951,0.735277,0.686831,0.669970,0.735186,0.687023,0.670034


TrainOutput(global_step=4150, training_loss=0.5046571988944548, metrics={'train_runtime': 10685.2158, 'train_samples_per_second': 49.625, 'train_steps_per_second': 0.388, 'total_flos': 3.487865927616e+16, 'train_loss': 0.5046571988944548, 'epoch': 50.0})

In [20]:
results = trainer.evaluate()
print(results)

{'eval_loss': 0.6881137490272522, 'eval_accuracy': 0.6547921967769296, 'eval_precision': 0.5974304068522484, 'eval_recall': 0.9473684210526315, 'eval_f1': 0.7327642810242941, 'eval_precision_macro': 0.7354498973036752, 'eval_recall_macro': 0.6550401427297056, 'eval_f1_macro': 0.6226695656618477, 'eval_precision_weighted': 0.7355669621810725, 'eval_recall_weighted': 0.6547921967769296, 'eval_f1_weighted': 0.6225761859202341, 'eval_runtime': 9.0605, 'eval_samples_per_second': 130.126, 'eval_steps_per_second': 1.104, 'epoch': 50.0}


In [21]:
trainer.save_model("/content/drive/MyDrive/Colab Notebooks/major projectr/sarcasm_RoBERTa_model")
tokenizer.save_pretrained("/content/drive/MyDrive/Colab Notebooks/major projectr/sarcasm_RoBERTa_model")

('/content/drive/MyDrive/Colab Notebooks/major projectr/sarcasm_RoBERTa_model/tokenizer_config.json',
 '/content/drive/MyDrive/Colab Notebooks/major projectr/sarcasm_RoBERTa_model/special_tokens_map.json',
 '/content/drive/MyDrive/Colab Notebooks/major projectr/sarcasm_RoBERTa_model/vocab.json',
 '/content/drive/MyDrive/Colab Notebooks/major projectr/sarcasm_RoBERTa_model/merges.txt',
 '/content/drive/MyDrive/Colab Notebooks/major projectr/sarcasm_RoBERTa_model/added_tokens.json',
 '/content/drive/MyDrive/Colab Notebooks/major projectr/sarcasm_RoBERTa_model/tokenizer.json')

In [22]:
from transformers import pipeline

clf = pipeline("text-classification", model="sarcasm_bert_model")

clf("Government promises transparency, deletes all records")

OSError: sarcasm_bert_model is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `huggingface-cli login` or by passing `token=<your_token>`

In [ ]:
from sklearn.metrics import classification_report

preds_output = trainer.predict(val_dataset)
preds = np.argmax(preds_output.predictions, axis=1)

print(classification_report(val_labels, preds, target_names=["non-sarcastic", "sarcastic"]))